In [2]:
import requests
import time
import os
import json
import pandas as pd
from IPython.display import clear_output

# Configurações
BUFFER_SIZE = 200  # número de apps antes de salvar no arquivo
DATA_DIR = "../data/raw"
BATCH_FILE_PREFIX = os.path.join(DATA_DIR, "games_batch")
CHECKPOINT_FILE = os.path.join(DATA_DIR, "checkpoint.csv")

# Buffers
game_data_buffer = []
processed_apps_buffer = []

# Sessão persistente
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

# --- Funções --- #

def ensure_data_dir():
    """Garante que o diretório data existe"""
    if not os.path.exists(DATA_DIR):
        os.makedirs(DATA_DIR)
        print(f"📁 Diretório criado: {DATA_DIR}")

def load_checkpoint():
    """Carrega lista de app_ids processados a partir do checkpoint CSV"""
    ensure_data_dir()
    if os.path.exists(CHECKPOINT_FILE):
        try:
            df = pd.read_csv(CHECKPOINT_FILE)
            processed = df['app_id'].tolist()
            print(f"📂 Checkpoint carregado: {len(processed)} apps processados anteriormente.")
            return processed
        except Exception as e:
            print(f"❌ Erro ao carregar checkpoint: {e}")
            return []
    return []

def save_buffers():
    """
    Salva buffers de jogos em Parquet e checkpoint em CSV.
    - Converte todas as colunas para string para evitar erros de tipagem.
    """
    global game_data_buffer, processed_apps_buffer

    # --- Salvar jogos em Parquet ---
    if game_data_buffer:
        try:
            df_games = pd.DataFrame(game_data_buffer)

            # Converter todas as colunas para string
            for col in df_games.columns:
                df_games[col] = df_games[col].apply(lambda x: json.dumps(x, ensure_ascii=False) 
                                                    if isinstance(x, (dict, list)) 
                                                    else str(x) if x is not None else "")

            timestamp = int(time.time())
            parquet_file = f"{BATCH_FILE_PREFIX}_{timestamp}.parquet"

            df_games.to_parquet(parquet_file, index=False, engine='pyarrow')
            print(f"💾 Batch salvo: {parquet_file} ({len(game_data_buffer)} registros)")

            game_data_buffer = []
        except Exception as e:
            print(f"❌ Erro ao salvar batch Parquet: {e}")
            raise e

    # --- Salvar checkpoint em CSV ---
    if processed_apps_buffer:
        try:
            df_checkpoint = pd.DataFrame({'app_id': [str(a) for a in processed_apps_buffer]})
            header = not os.path.exists(CHECKPOINT_FILE)
            df_checkpoint.to_csv(CHECKPOINT_FILE, mode='a', header=header, index=False)
            print(f"📝 Checkpoint atualizado: {len(processed_apps_buffer)} apps")
            processed_apps_buffer = []
        except Exception as e:
            print(f"❌ Erro ao salvar checkpoint CSV: {e}")

    clear_output(wait=True)

def save_game_data(game_data):
    """Adiciona app ao buffer de dados"""
    global game_data_buffer
    game_data_buffer.append(game_data)

def get_all_steam_apps():
    """Retorna lista de todos os app_ids da Steam"""
    url = "https://api.steampowered.com/ISteamApps/GetAppList/v2/"
    try:
        response = session.get(url)
        response.raise_for_status()
        data = response.json()
        return [app['appid'] for app in data['applist']['apps']]
    except Exception as e:
        print(f"❌ Erro ao buscar lista de apps: {e}")
        return []

def get_game_details(app_id, retries=10):
    """Obtém full_data de um jogo da Steam"""
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}&l=en&cc=US"
    
    for attempt in range(retries):
        try:
            response = session.get(url)
            response.raise_for_status()
            data = response.json()
            
            game_data = data.get(str(app_id), {})
            if game_data.get("success"):
                return game_data["data"]
            
            print(f"⚠️ AppID {app_id} não possui dados disponíveis.")
            return None
            
        except requests.exceptions.HTTPError as e:
            if response.status_code == 429:
                print(f"⏳ 429 Too Many Requests para AppID {app_id}, aguardando 100s...")
                time.sleep(100)
                continue

            if attempt == (retries - 1):
                raise e
            return None
        except requests.exceptions.RequestException as e:
            print(f"🚨 Erro de requisição para AppID {app_id}: {e}, aguardando 30s...")
            time.sleep(30)
            if attempt == (retries -1):
                raise e
            continue
    
    return None

# --- Main --- #

def main():
    ensure_data_dir()
    all_apps = get_all_steam_apps()
    if not all_apps:
        print("❌ Nenhum app encontrado. Encerrando.")
        return

    processed_apps = load_checkpoint()
    apps_to_process = [a for a in all_apps if a not in processed_apps]
    
    print(f"⚡ Total de apps a processar: {len(apps_to_process)}")
    print(f"📁 Dados serão salvos em: {DATA_DIR}")

    for i, app_id in enumerate(apps_to_process, start=1):
        print(f"🔎 Processando AppID {app_id} ({i}/{len(apps_to_process)})")
        
        details = get_game_details(app_id)
        
        # Sempre salva o app_id no checkpoint, mesmo que não exista
        processed_apps_buffer.append(app_id)
        
        if details:
            save_game_data(details)

        gd_count = len(game_data_buffer)
        print(f"📊 Progresso Buffer: {gd_count}/{BUFFER_SIZE}")
        if gd_count >= BUFFER_SIZE:
            save_buffers()

        time.sleep(2)  # evita sobrecarga da API

    # Salvar o que sobrou no buffer
    save_buffers()
    print(f"🎉 Extração concluída! Dados salvos em: {DATA_DIR}")

if __name__ == "__main__":
    main()

🎉 Extração concluída! Dados salvos em: ../data/raw


In [1]:
import pandas as pd
import os
import glob


def read_all_parquets(data_dir="../data/raw"):
    parquet_files = glob.glob(os.path.join(data_dir, "games_batch_*.parquet"))
    
    if not parquet_files:
        print("❌ Nenhum arquivo Parquet encontrado")
        return None
    
    dataframes = []
    for file in parquet_files:
        df = pd.read_parquet(file)
        dataframes.append(df)
        print(f"📖 Lido: {os.path.basename(file)} - {len(df)} registros")
    
    # Combinar todos os DataFrames
    combined_df = pd.concat(dataframes, ignore_index=True)
    print(f"\n🎯 Total combinado: {len(combined_df)} registros")
    return combined_df

df = read_all_parquets()

📖 Lido: games_batch_1759566025.parquet - 200 registros
📖 Lido: games_batch_1759549456.parquet - 200 registros
📖 Lido: games_batch_1759561757.parquet - 200 registros
📖 Lido: games_batch_1759581386.parquet - 200 registros
📖 Lido: games_batch_1759555460.parquet - 200 registros
📖 Lido: games_batch_1759538123.parquet - 200 registros
📖 Lido: games_batch_1759558318.parquet - 200 registros
📖 Lido: games_batch_1759591612.parquet - 200 registros
📖 Lido: games_batch_1759584545.parquet - 200 registros
📖 Lido: games_batch_1759575001.parquet - 200 registros
📖 Lido: games_batch_1759534533.parquet - 200 registros
📖 Lido: games_batch_1759560803.parquet - 200 registros
📖 Lido: games_batch_1759567597.parquet - 200 registros
📖 Lido: games_batch_1759537139.parquet - 200 registros
📖 Lido: games_batch_1759583714.parquet - 200 registros
📖 Lido: games_batch_1759570746.parquet - 200 registros
📖 Lido: games_batch_1759578292.parquet - 200 registros
📖 Lido: games_batch_1759569874.parquet - 200 registros
📖 Lido: ga

In [1]:
df.head()

NameError: name 'df' is not defined

In [6]:
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import numpy as np

# Configurar visualizações
pd.set_option('display.max_columns', None)  # Mostrar todas as colunas:cite[5]
plt.style.use('seaborn-v0_8')

print("📊 ANÁLISE EXPLORATÓRIA DOS DADOS")
print("=" * 50)

# Carregar os dados
df_games_detail = pd.read_csv('steam_games_detailed.csv', encoding='utf-8')
df_user_review = pd.read_csv('steam_user_reviews.csv', encoding='utf-8')

print("1. PRIMEIRAS LINHAS DOS DADOS DE JOGOS:")
print(df_games_detail.head(3))

print("\n2. PRIMEIRAS LINHAS DAS REVIEWS:")
print(df_user_review.head(3))

print("\n3. INFORMAÇÕES SOBRE OS DATAFRAMES:")
print(f"Detalhes de jogos: {df_games_detail.shape} linhas e colunas")
print(f"Reviews de usuários: {df_user_review.shape} linhas e colunas")

ModuleNotFoundError: No module named 'torch'

In [ ]:
print("\n🔗 PREPARAÇÃO DOS DADOS")
print("=" * 50)

# Verificar dados faltantes
print("Dados faltantes em df_games_detail:")
print(df_games_detail.isnull().sum())

print("\nDados faltantes em df_user_review:")
print(df_user_review.isnull().sum())

# Criar uma amostra menor para desenvolvimento (opcional)
# Vamos trabalhar com uma amostra de 50,000 reviews para velocidade
df_user_review_sample = df_user_review.sample(n=min(50000, len(df_user_review)), random_state=42)

# Unir os dados
print("\nUnindo os DataFrames...")
df_combined = pd.merge(df_user_review_sample, df_games_detail, on='app_id', how='left')

print(f"DataFrame combinado: {df_combined.shape}")

# Criar features básicas para o modelo
print("\nCriando features para o modelo...")

# 1. Feature de engajamento do usuário (playtime total)
df_combined['engagement'] = np.log1p(df_combined['author_playtime_forever'])

# 2. Converter voted_up para numérico (nosso target)
df_combined['liked'] = df_combined['voted_up'].astype(int)


print("Features criadas: engagement, liked")
print(f"Distribuição de 'liked': {df_combined['liked'].value_counts().to_dict()}")


🔗 PREPARAÇÃO DOS DADOS
Dados faltantes em df_games_detail:
app_id               0
name                 0
type                 0
recommendations      0
is_free              0
genres               0
categories           0
developers           0
publishers           0
release_date         0
short_description    0
dtype: int64

Dados faltantes em df_user_review:
app_id                      0
review_id                   0
author_steamid              0
author_playtime_forever     0
author_playtime_2weeks      0
language                    0
review_text                51
voted_up                    0
votes_up                    0
votes_funny                 0
timestamp_created           0
timestamp_updated           0
dtype: int64

Unindo os DataFrames...
DataFrame combinado: (9800, 22)

Criando features para o modelo...
Features criadas: engagement, liked
Distribuição de 'liked': {1: 8032, 0: 1768}


In [ ]:
df_combined.columns

Index(['app_id', 'review_id', 'author_steamid', 'author_playtime_forever',
       'author_playtime_2weeks', 'language', 'review_text', 'voted_up',
       'votes_up', 'votes_funny', 'timestamp_created', 'timestamp_updated',
       'name', 'type', 'recommendations', 'is_free', 'genres', 'categories',
       'developers', 'publishers', 'release_date', 'short_description',
       'engagement', 'liked'],
      dtype='object')

In [ ]:
import torch  # Importa o PyTorch, uma biblioteca para computação numérica e aprendizado de máquina.
from sklearn.preprocessing import LabelEncoder  # Importa o LabelEncoder para codificar variáveis categóricas.
from torch.utils.data import Dataset, DataLoader  # Importa classes para manipulação de dados em PyTorch.
import torch.nn as nn  # Importa o módulo de redes neurais do PyTorch.
import torch.optim as optim  # Importa otimizadores do PyTorch.

# Carregar os dados
df = df_combined  # Atribui o DataFrame combinado à variável df.
print("DataFrame carregado:\n", df.head())  # Exibe as primeiras linhas do DataFrame

# Codificar variáveis categóricas
user_encoder = LabelEncoder()  # Cria um codificador para a coluna 'author_steamid'.
df['user'] = user_encoder.fit_transform(df['author_steamid'])  # Codifica a coluna 'author_steamid' e armazena em 'user'.
print("Coluna 'user' codificada:\n", df[['author_steamid', 'user']].head())  # Exibe a codificação dos usuários

game_encoder = LabelEncoder()  # Cria um codificador para a coluna 'name'.
df['game'] = game_encoder.fit_transform(df['name'])  # Codifica a coluna 'name' e armazena em 'game'.
print("Coluna 'game' codificada:\n", df[['name', 'game']].head())  # Exibe a codificação dos jogos

# Selecionar colunas relevantes
df = df[['user', 'game', 'liked']]  # Mantém apenas as colunas 'user', 'game' e 'liked' no DataFrame.
print("DataFrame final selecionado:\n", df.head())  # Exibe o DataFrame final

class RecommendationDataset(Dataset):  # Define uma classe personalizada para o conjunto de dados.
    def __init__(self, df):
        self.users = torch.tensor(df['user'].values, dtype=torch.long)  # Converte os usuários para tensor de inteiros.
        print("Tensor de usuários:", self.users[:5])  # Print dos primeiros 5 usuários
        self.games = torch.tensor(df['game'].values, dtype=torch.long)  # Converte os jogos para tensor de inteiros.
        print("Tensor de jogos:", self.games[:5])  # Print dos primeiros 5 jogos
        self.labels = torch.tensor(df['liked'].values, dtype=torch.float32)  # Converte os rótulos para tensor de ponto flutuante.
        print("Tensor de labels:", self.labels[:5])  # Print dos primeiros 5 labels

    def __len__(self):
        length = len(self.users)  # Retorna o número de amostras no conjunto de dados.
        print("Tamanho do dataset:", length)  # Print do tamanho do dataset
        return length

    def __getitem__(self, idx):
        sample = (self.users[idx], self.games[idx], self.labels[idx])  # Retorna uma amostra específica com base no índice.
        print(f"Amostra {idx}: {sample}")  # Print da amostra acessada
        return sample

class RecommenderModel(nn.Module):  # Define um modelo de recomendação baseado em redes neurais.
    def __init__(self, num_users, num_games, embedding_dim=50):
        super(RecommenderModel, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)  # Cria uma camada de embedding para usuários.
        print(f"Embedding de usuários criado com shape ({num_users}, {embedding_dim})")
        self.game_embedding = nn.Embedding(num_games, embedding_dim)  # Cria uma camada de embedding para jogos.
        print(f"Embedding de jogos criado com shape ({num_games}, {embedding_dim})")
        self.fc = nn.Linear(embedding_dim, 1)  # Cria uma camada totalmente conectada para previsão.
        print(f"Camada fully connected criada com input dim {embedding_dim} e output dim 1")

    def forward(self, user, game):
        user_emb = self.user_embedding(user)  # Obtém o embedding do usuário.
        print("Embedding dos usuários:", user_emb[:5])  # Print dos primeiros 5 embeddings de usuário
        game_emb = self.game_embedding(game)  # Obtém o embedding do jogo.
        print("Embedding dos jogos:", game_emb[:5])  # Print dos primeiros 5 embeddings de jogo
        x = user_emb * game_emb  # Realiza o produto elemento a elemento entre os embeddings.
        print("Produto elemento a elemento:", x[:5])  # Print dos primeiros 5 produtos
        x = self.fc(x)  # Passa o resultado pela camada totalmente conectada.
        print("Saída da camada FC:", x[:5])  # Print dos primeiros 5 outputs
        return x  # Retorna a previsão.

# Dados de exemplo
num_users = df['user'].nunique()  # Define o número de usuários.
print("Número de usuários:", num_users)
num_games = df['game'].nunique()  # Define o número de jogos.
print("Número de jogos:", num_games)
embedding_dim = 10  # Define a dimensão dos embeddings.
print("Dimensão dos embeddings:", embedding_dim)
model = RecommenderModel(num_users, num_games, embedding_dim)  # Cria uma instância do modelo de recomendação.

# Dados de entrada
users = torch.tensor(df['user'].values, dtype=torch.long)  # Exemplo de IDs de usuários.
print("Tensor users:", users[:5])
games = torch.tensor(df['game'].values, dtype=torch.long)  # Exemplo de IDs de jogos.
print("Tensor games:", games[:5])
labels = torch.tensor(df['liked'].values, dtype=torch.float32).unsqueeze(1)  # Rótulos de preferência, com dimensão extra.
print("Tensor labels:", labels[:5])

# Inicializando a função de perda e o otimizador
criterion = nn.BCEWithLogitsLoss()  # Define a função de perda para classificação binária com logits.
print("Função de perda:", criterion)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Define o otimizador Adam com taxa de aprendizado de 0.001.
print("Otimizador:", optimizer)

# Treinamento
model.train()  # Coloca o modelo em modo de treinamento.
num_epochs = 5

for epoch in range(1, num_epochs + 1):
    optimizer.zero_grad()  # zera os gradientes

    # Forward pass
    output = model(users, games)
    
    # Calcula a loss
    loss = criterion(output, labels)
    
    # Backward pass
    loss.backward()
    
    # Atualiza os parâmetros
    optimizer.step()
    
    print(f"Época {epoch}/{num_epochs} - Loss: {loss.item()}")



DataFrame carregado:
     app_id  review_id     author_steamid  author_playtime_forever  \
0   381210  205574694  76561199305984475                    11.63   
1  3240220  205551853  76561199785674403                    50.33   
2  1004640  205576843  76561198821451341                     1.30   
3  1326470  205547624  76561199821213859                    61.78   
4   230410  205555622  76561199877373204                   111.12   

   author_playtime_2weeks  language  \
0                     0.0   english   
1                     0.0   english   
2                     0.0   english   
3                     0.0   russian   
4                     0.0  schinese   

                                         review_text  voted_up  votes_up  \
0                                         very scare      True         0   
1  4444444444444444444444444444444444444444444444...      True         0   
2  The Good\n\n- Modern UI\n- Includes the origin...      True         0   
3  явный прогресс по сра

In [ ]:

# Treinamento
model.train()  # Coloca o modelo em modo de treinamento.
num_epochs = 5 

for epoch in range(1, num_epochs + 1):
    optimizer.zero_grad()  # zera os gradientes

    # Forward pass
    output = model(users, games)
    
    # Calcula a loss
    loss = criterion(output, labels)
    
    # Backward pass
    loss.backward()
    
    # Atualiza os parâmetros
    optimizer.step()
    
    print(f"Época {epoch}/{num_epochs} - Loss: {loss.item()}")
    print("Primeiras 5 previsões:", torch.sigmoid(output[:5]).detach().numpy())  # sigmoid para transformar logits em probabilidade

Embedding dos usuários: tensor([[ 1.1555, -0.2600, -0.8533, -0.0532, -1.1813, -1.2453, -0.1866, -0.5550,
         -0.8006,  0.1238],
        [-1.3085, -1.0899,  0.8654, -0.9673,  0.2802, -0.5326,  0.2772,  0.1546,
         -0.5975, -0.4319],
        [ 0.4711,  0.0417, -0.1711,  0.7715,  0.7116,  0.6665,  0.0149,  0.0123,
         -0.9615,  0.1772],
        [ 0.7218,  1.0854,  0.4930,  0.1773,  0.2458, -0.0629,  0.4787,  0.4091,
         -0.1785, -0.7965],
        [-0.9765, -2.5011,  0.6310, -2.3243,  0.0351,  0.9107, -0.0470, -0.0160,
         -2.5618,  0.7201]], grad_fn=<SliceBackward0>)
Embedding dos jogos: tensor([[ 0.2936,  1.6800, -1.6352, -0.6687,  1.3687,  0.5568, -0.2113,  0.3454,
         -1.2026,  0.8851],
        [ 0.7329,  1.3993, -1.1714,  0.0930,  0.0422, -0.5337,  1.1093,  0.1160,
         -0.1133,  0.4228],
        [ 0.5255,  1.9597,  0.4132, -0.3205, -0.7640,  0.3057, -0.3949,  0.2692,
         -0.8030,  0.9081],
        [ 0.5911,  0.7768, -0.3365, -1.9433, -1.7222, -1

In [ ]:
user_map = pd.DataFrame({
    'steam_id': user_encoder.classes_,
    'user_id': range(len(user_encoder.classes_))
})
print(user_map.head(-10))

               steam_id  user_id
0     76561197960279930        0
1     76561197960310043        1
2     76561197960312749        2
3     76561197960414917        3
4     76561197960432447        4
...                 ...      ...
9530  76561199888398134     9530
9531  76561199888495121     9531
9532  76561199888704887     9532
9533  76561199888817374     9533
9534  76561199889642344     9534

[9535 rows x 2 columns]


In [ ]:
def recommend(user_id, top_n=5):
    model.eval()
    user_tensor = torch.tensor([user_id], dtype=torch.long)
    all_game_ids = torch.tensor(range(df['game'].nunique()), dtype=torch.long)

    # Descarta jogos que o usuário já jogou
    played_games = df[df['user'] == user_id]['game'].values
    mask = ~torch.isin(all_game_ids, torch.tensor(played_games))
    candidate_game_ids = all_game_ids[mask]

    with torch.no_grad():
        scores = model(user_tensor.repeat(len(candidate_game_ids)), candidate_game_ids).squeeze()

    top_games = scores.argsort(descending=True)[:top_n]
    return game_encoder.inverse_transform(candidate_game_ids[top_games].numpy())


# Exemplo de recomendação
recommended_games = recommend(user_id=1)
print("Jogos recomendados:", recommended_games)


Embedding dos usuários: tensor([[ 7.8081e-02, -7.4530e-01, -7.3419e-01,  1.4908e-01,  1.5542e-01,
          1.2839e+00, -5.6343e-01, -1.5816e+00, -4.9474e-01,  4.9900e-05],
        [ 7.8081e-02, -7.4530e-01, -7.3419e-01,  1.4908e-01,  1.5542e-01,
          1.2839e+00, -5.6343e-01, -1.5816e+00, -4.9474e-01,  4.9900e-05],
        [ 7.8081e-02, -7.4530e-01, -7.3419e-01,  1.4908e-01,  1.5542e-01,
          1.2839e+00, -5.6343e-01, -1.5816e+00, -4.9474e-01,  4.9900e-05],
        [ 7.8081e-02, -7.4530e-01, -7.3419e-01,  1.4908e-01,  1.5542e-01,
          1.2839e+00, -5.6343e-01, -1.5816e+00, -4.9474e-01,  4.9900e-05],
        [ 7.8081e-02, -7.4530e-01, -7.3419e-01,  1.4908e-01,  1.5542e-01,
          1.2839e+00, -5.6343e-01, -1.5816e+00, -4.9474e-01,  4.9900e-05]])
Embedding dos jogos: tensor([[-0.5703,  0.9915, -0.4600, -1.3673, -0.4348, -0.9281,  0.1962, -1.2849,
         -0.2098,  1.4298],
        [-0.0403, -0.6878, -1.3022, -1.0505,  0.2191, -1.7656,  0.4285, -1.4553,
         -0.0574, -

In [ ]:
AP_KEY = "BB450D3100336BFB4B540F9A6728744A"